In [1]:
from pathlib import Path
import requests
import hashlib
import zipfile
from io import BytesIO
from datetime import date, timedelta
import time
from time import perf_counter
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
def fetch_gdelt_events(url, save_dir):
    # Create the save directory if it doesn't exist
    Path(save_dir).mkdir(parents=True, exist_ok=True)

    # Download the file
    response = requests.get(url, verify=False)

    if response.status_code == 200:
        # Extract the filename from the URL
        filename = url.split('/')[-1]
        file_path = Path(save_dir) / filename

        # Save the downloaded content to a file
        # save the content of the zip file instead of the zip file itself
        
        with open(file_path, 'wb') as f:
            f.write(response.content)

        print(f"Downloaded and saved: {file_path}")

    else:
        print(f"Failed to download file. Status code: {response.status_code}")

In [3]:
# construct 2016 urls
base_url = "https://data.gdeltproject.org/gdeltv2/20160101000000.mentions.CSV.zip"

In [4]:
fetch_gdelt_events(base_url, r"D:\gdelt_data\2016_mentions")

Downloaded and saved: D:\gdelt_data\2016_mentions\20160101000000.mentions.CSV.zip


In [5]:
def make_gdelt_urls(year, file_type="export"):
    dates = pd.date_range(
        start=f"{year}-01-01 00:00:00",
        end=f"{year}-12-31 23:45:00",
        freq="15min"
    )

    return [
        f"https://data.gdeltproject.org/gdeltv2/{dt:%Y%m%d%H%M%S}.{file_type}.CSV.zip"
        for dt in dates
    ]

failed_files = []

def download_file(url, save_dir, overwrite=False):
    
    global failed_files

    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    filename = url.split("/")[-1]
    file_path = save_dir / filename

    if file_path.exists() and not overwrite:
        return f"Skipped: {filename}"

    try:
        response = requests.get(url, verify=False, timeout=60)

        if response.status_code == 200:
            with open(file_path, "wb") as f:
                f.write(response.content)
            return f"Downloaded: {filename}"
        else:
            failed_files.append(filename)
            return f"Failed {response.status_code}: {filename}"

    except Exception as e:
        return f"Error: {filename} | {e}"


def bulk_download_gdelt(year, save_dir, file_type="export", max_workers=8):
    urls = make_gdelt_urls(year, file_type=file_type)

    print(f"Downloading {len(urls):,} files for {year}...")
    print(f"Saving to: {save_dir}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(download_file, url, save_dir)
            for url in urls
        ]

        for n, future in enumerate(as_completed(futures), start=1):
            result = future.result()

            if n % 100 == 0:
                print(f"{n:,}/{len(urls):,} completed")

            if result.startswith(("Failed", "Error")):
                print(result)

    print("Done.")

#should add functionality to save names of failed downloads to a file for later review and retrying.
# up to 1029030000

In [6]:
"""
bulk_download_gdelt(
    year=2025,
    save_dir=r"D:\gdelt_data\2025_mentions",
    file_type="mentions",
    max_workers=8
)
"""

'\nbulk_download_gdelt(\n    year=2025,\n    save_dir=r"D:\\gdelt_data\x825_mentions",\n    file_type="mentions",\n    max_workers=8\n)\n'

In [7]:
for year in range(2016, 2026):
    bulk_download_gdelt(
        year=year,
        save_dir=f"D:\\gdelt_data\\{year}_mentions",
        file_type="mentions",
        max_workers=8
    )

with open("failed_files.txt", "w") as f:
    f.write("\n".join(failed_files))

Saving to: D:\gdelt_data\2016_mentions
100/35,136 completed
200/35,136 completed
300/35,136 completed
400/35,136 completed
500/35,136 completed
600/35,136 completed
700/35,136 completed
800/35,136 completed
900/35,136 completed
Failed 404: 20160107203000.mentions.CSV.zip
1,000/35,136 completed
Failed 404: 20160108014500.mentions.CSV.zip
1,100/35,136 completed
1,200/35,136 completed
1,300/35,136 completed
Failed 404: 20160112014500.mentions.CSV.zip
Failed 404: 20160112001500.mentions.CSV.zip
1,400/35,136 completed
1,500/35,136 completed
1,600/35,136 completed
1,700/35,136 completed
1,800/35,136 completed
1,900/35,136 completed
2,000/35,136 completed
2,100/35,136 completed
2,200/35,136 completed
Failed 404: 20160122013000.mentions.CSV.zip
2,300/35,136 completed
2,400/35,136 completed
2,500/35,136 completed
2,600/35,136 completed
Failed 404: 20160124234500.mentions.CSV.zip
2,700/35,136 completed
2,800/35,136 completed
2,900/35,136 completed
3,000/35,136 completed
3,100/35,136 completed
Fa

In [ ]:
failed_files